In [9]:
import glob
import gzip
import json
import pandas as pd

# 아직 안 쓴 파일 1개만 샘플로 확인
file_path = "/content/drive/MyDrive/EcoTracing/data/raw/instance_usage-000000000004.json.gz"

data = []
with gzip.open(file_path, 'rt') as f:
    for i, line in enumerate(f):
        if i >= 10000:
                break
        row = json.loads(line)
        data.append(row)

df = pd.DataFrame(data)
print(df["duration"].describe())
print(df["duration"].value_counts().head(10))

KeyError: 'duration'

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import os
import yaml
import pickle
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Config 로드
CONFIG_PATH = "/content/config.yaml"
with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

DRIVE_ROOT = f"/content/drive/MyDrive/{config['project_name']}"
PROCESSED_PATH = os.path.join(DRIVE_ROOT, config['paths']['processed_data'],
    config['paths']['processed_file'])

# 데이터 로드
df = pd.read_parquet(PROCESSED_PATH)
print(f"Total rows: {len(df):,}")

# ============================================================
# 비교할 피처 조합
# ============================================================

feature_sets = {
    "cpu+duration only": ["cpu", "duration"],
    "cpu+memory+duration": ["cpu", "memory", "duration"],
    "cpu+duration+hour": ["cpu", "duration", "hour"],
    "all (cpu+memory+duration+hour)": ["cpu", "memory", "duration", "hour"],
}

target = "energy_kwh"
results = []

for name, features in feature_sets.items():
    X = df[features]
    y = df[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=config['model']['test_size'],
        random_state=config['model']['random_state']
    )

    model = RandomForestRegressor(
        n_estimators=100,
        random_state=config['model']['random_state'],
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

    results.append({
        "features": name,
        "rmse":     rmse,
        "mae":      mae,
        "r2":       r2,
        "mape":     mape,
    })

    print(f"\n[{name}]")
    print(f"  RMSE: {rmse:.6f} | MAE: {mae:.6f} | R2: {r2:.6f} | MAPE: {mape:.2f}%")

# 결과 정리
df_results = pd.DataFrame(results)
print("\n\n=== 최종 비교 ===")
print(df_results.to_string(index=False))

Total rows: 19,523,808

[cpu+duration only]
  RMSE: 0.000011 | MAE: 0.000006 | R2: 0.999997 | MAPE: 0.04%

[cpu+memory+duration]
  RMSE: 0.000002 | MAE: 0.000000 | R2: 1.000000 | MAPE: 0.00%

[cpu+duration+hour]
  RMSE: 0.000010 | MAE: 0.000006 | R2: 0.999997 | MAPE: 0.04%

[all (cpu+memory+duration+hour)]
  RMSE: 0.000002 | MAE: 0.000000 | R2: 1.000000 | MAPE: 0.00%


=== 최종 비교 ===
                      features     rmse          mae       r2     mape
             cpu+duration only 0.000011 6.089077e-06 0.999997 0.038729
           cpu+memory+duration 0.000002 6.541414e-08 1.000000 0.001348
             cpu+duration+hour 0.000010 5.720456e-06 0.999997 0.036444
all (cpu+memory+duration+hour) 0.000002 7.076668e-08 1.000000 0.001408
